# 09 — psfFlux / scienceFlux / templateFlux for Gaia DR3–matched Fink objects

## Purpose

This notebook is a **restricted version of `07_psfFlux_scienceFlux_templateFlux.ipynb`**:
it displays the raw fluxes in nJy for the subset of Fink diaObjects that were
successfully joined to a Gaia DR3 source in `08b_matchFinkAlertswithGaiaDR3.ipynb`.

The three flux quantities compared are:

- `psfFlux`      — difference-image PSF flux (can be negative)
- `scienceFlux`  — PSF flux on the science calexp (positive for real sources)
- `templateFlux` — PSF flux on the template image

**No normalisation** is applied.  The Gaia DR3 match provides additional
context: Gaia G magnitude, stability flag, and Fink group are shown in the
figure titles.

## Inputs

| File | Produced by |
|------|-------------|
| `data_FINK_BLOCK_LC_03/src_joined_butler.parquet` (or `consdb`) | `03_associateFinkAlerts-RubinVisits.ipynb` |
| `data_FINK_BLOCK_LC_08b/fink_diaobj_gaia_join_matched.csv`      | `08b_matchFinkAlertswithGaiaDR3.ipynb`     |

---

- **Author:** Sylvie Dagoret-Campagne  
- **Affiliation:** IJCLab / IN2P3 / CNRS  
- **Follows:** `07_psfFlux_scienceFlux_templateFlux.ipynb` and `08b_matchFinkAlertswithGaiaDR3.ipynb`
- **Creation date:** 2026-04-11  
- **Subject:** Fink alert broker — Rubin LSST DIA photometry diagnostics on Gaia DR3 stars


## 1. Imports & configuration

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from astropy.time import Time

warnings.filterwarnings("ignore")

print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

In [ ]:
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline")

In [ ]:
# ── Notebook tag & paths ──────────────────────────────────────────────────────
NB_TAG = "FINK_BLOCK_LC_09"

# Flux data (same source as notebook 07)
DIR_DATA_FLUX = "data_FINK_BLOCK_LC_03"
FILE_BUTLER = os.path.join(DIR_DATA_FLUX, "src_joined_butler.parquet")
FILE_CONSDB = os.path.join(DIR_DATA_FLUX, "src_joined_consdb.parquet")

# Gaia DR3 matched catalogue from notebook 08b
DIR_DATA_GAIA08b = "data_FINK_BLOCK_LC_08b"
FILE_GAIA_MATCHED = os.path.join(DIR_DATA_GAIA08b, "fink_diaobj_gaia_join_matched.csv")

DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_FIGS, exist_ok=True)

N_TOP = 50  # top-N Gaia-matched objects by alert count

BANDS = list("ugrizy")
BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)

# ── Column names (Fink alert schema) ─────────────────────────────────────────
COL_OBJ = "r:diaObjectId"
COL_SRC = "r:diaSourceId"
COL_MJD = "r:midpointMjdTai"
COL_BAND = "r:band"
COL_PSF = "r:psfFlux"
COL_PSFERR = "r:psfFluxErr"

SCIENCE_FLUX_CANDIDATES = [
    "r:scienceFlux",
    "r:psfScience",
    "r:psfScienceFlux",
    "scienceFlux",
    "psfScience",
    "psfScienceFlux",
]
SCIENCE_ERR_CANDIDATES = [
    "r:scienceSigma",
    "r:scienceFluxErr",
    "r:psfScienceSigma",
    "r:psfScienceFluxErr",
    "scienceSigma",
    "scienceFluxErr",
    "psfScienceSigma",
]
TEMPLATE_FLUX_CANDIDATES = [
    "psfTemplateFlux",
    "r:templateFlux",
    "r:template",
    "templateFlux",
    "template",
]
TEMPLATE_ERR_CANDIDATES = [
    "r:templateSigma",
    "r:templateFluxErr",
    "r:psfTemplateSigma",
    "r:psfTemplateFluxErr",
    "templateSigma",
    "templateFluxErr",
    "psfTemplateSigma",
]


def savefig(name):
    """Save current figure as PDF and PNG in DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  → saved {name}.{{pdf,png}}")


print(f"Flux data dir    : {os.path.abspath(DIR_DATA_FLUX)}")
print(f"Gaia match file  : {os.path.abspath(FILE_GAIA_MATCHED)}")
print(f"Figure directory : {os.path.abspath(DIR_FIGS)}")
print(f"N_TOP            : {N_TOP}")

## 2. Load flux data (src parquet from notebook 03)

In [ ]:
if os.path.exists(FILE_BUTLER):
    df_src = pd.read_parquet(FILE_BUTLER)
    src_label = "butler"
    print(f"Loaded butler file : {FILE_BUTLER}")
elif os.path.exists(FILE_CONSDB):
    df_src = pd.read_parquet(FILE_CONSDB)
    src_label = "consdb"
    print(f"Butler file not found. Loaded consDb file: {FILE_CONSDB}")
else:
    raise FileNotFoundError(
        f"Neither {FILE_BUTLER} nor {FILE_CONSDB} found.\n"
        "Run notebook 03_associateFinkAlerts-RubinVisits.ipynb first."
    )

print(f"Shape: {df_src.shape[0]:,} rows × {df_src.shape[1]} columns")

## 3. Load Gaia DR3 matched catalogue (from notebook 08b)

This catalogue contains one row per diaObject with the Gaia DR3 `source_id`
already joined.  We use it to:

1. Build the list of `diaObjectId` values that have a confirmed Gaia DR3 match.
2. Attach Gaia metadata (G magnitude, Fink group, Gaia stability class) to the
   figure titles for scientific context.

In [ ]:
if not os.path.exists(FILE_GAIA_MATCHED):
    raise FileNotFoundError(
        f"{FILE_GAIA_MATCHED} not found.\nRun notebook 08b_matchFinkAlertswithGaiaDR3.ipynb first."
    )

df_gaia_matched = pd.read_csv(FILE_GAIA_MATCHED)

print("Gaia matched catalogue shape:", df_gaia_matched.shape)
print("Columns:", df_gaia_matched.columns.tolist())
print(df_gaia_matched.head(3).to_string())

In [ ]:
# Build lookup dict: diaObjectId → Gaia metadata row
# We index on the 'diaObjectId' column (plain integer, not the Fink 'r:diaObjectId' with prefix)
GAIA_ID_COL = "diaObjectId"

if GAIA_ID_COL not in df_gaia_matched.columns:
    raise KeyError(
        f"Column '{GAIA_ID_COL}' not found in {FILE_GAIA_MATCHED}.\n"
        f"Available columns: {df_gaia_matched.columns.tolist()}"
    )

# Set of Gaia-matched diaObjectIds (as strings for safe comparison)
gaia_matched_ids = set(df_gaia_matched[GAIA_ID_COL].astype(str).values)
print(f"Unique Gaia-matched diaObjectIds: {len(gaia_matched_ids)}")

# Lookup dict for per-object Gaia metadata
df_gaia_matched_idx = df_gaia_matched.set_index(GAIA_ID_COL)


def get_gaia_meta(obj_id):
    """Return a dict with Gaia metadata for a given diaObjectId, or empty dict."""
    key = int(str(obj_id))
    if key in df_gaia_matched_idx.index:
        row = df_gaia_matched_idx.loc[key]
        return {
            "group": row.get("group", "?"),
            "field": row.get("field", "?"),
            "G_mag": row.get("gaia_phot_g_mean_mag", np.nan),
            "gaia_stable": row.get("gaia_STABLE", None),
            "gaia_variable": row.get("gaia_is_gaia_variable", None),
            "gaia_status": row.get("gaia_status", "?"),
            "nDiaSrc": row.get("nDiaSources", None),
            "dr3_id": row.get("dr3_source_id", None),
        }
    return {}


print("Lookup function get_gaia_meta() defined.")

## 4. Schema inspection — detect scienceFlux and templateFlux columns

In [ ]:
df = df_src  # alias used throughout this notebook (same convention as notebook 07)

COL_SCI = None
COL_SCIERR = None
for candidate in SCIENCE_FLUX_CANDIDATES:
    if candidate in df.columns:
        COL_SCI = candidate
        print(f"Found science flux column  : '{COL_SCI}'")
        break
for candidate in SCIENCE_ERR_CANDIDATES:
    if candidate in df.columns:
        COL_SCIERR = candidate
        print(f"Found science flux err col : '{COL_SCIERR}'")
        break

COL_TEMP = None
COL_TEMPERR = None
for candidate in TEMPLATE_FLUX_CANDIDATES:
    if candidate in df.columns:
        COL_TEMP = candidate
        print(f"Found template flux column  : '{COL_TEMP}'")
        break
for candidate in TEMPLATE_ERR_CANDIDATES:
    if candidate in df.columns:
        COL_TEMPERR = candidate
        print(f"Found template flux err col : '{COL_TEMPERR}'")
        break

if COL_SCI is None:
    print("WARNING: No scienceFlux column found.")
if COL_TEMP is None:
    print("WARNING: No templateFlux column found.")

has_science = COL_SCI is not None
has_sci_err = COL_SCIERR is not None
has_template = COL_TEMP is not None
has_template_err = COL_TEMPERR is not None

print(f"\n  psfFlux      : {COL_PSF}")
print(f"  scienceFlux  : {COL_SCI}")
print(f"  scienceErr   : {COL_SCIERR}")
print(f"  templateFlux : {COL_TEMP}")
print(f"  templateErr  : {COL_TEMPERR}")

## 5. Filter flux data to Gaia DR3–matched diaObjects

We keep only the rows of the flux parquet whose `r:diaObjectId` appears in
the Gaia-matched catalogue produced by notebook 08b.

In [ ]:
# Cast parquet object column to string for safe set membership test
df["_oid_str"] = df[COL_OBJ].astype(str)

n_before = len(df)
df_gaia = df[df["_oid_str"].isin(gaia_matched_ids)].copy()
n_after = len(df_gaia)

print(f"Alerts before Gaia filter : {n_before:,}")
print(f"Alerts after  Gaia filter : {n_after:,}  ({100 * n_after / n_before:.1f}%)")
print(f"Unique diaObjects kept    : {df_gaia[COL_OBJ].nunique()}")

## 6. Rank Gaia-matched objects by alert count

In [ ]:
alert_counts = df_gaia.groupby(COL_OBJ).size().rename("n_alerts").sort_values(ascending=False)
print(f"Total Gaia-matched unique diaObjects : {len(alert_counts):,}")
print(f"Top {N_TOP}:")
print(alert_counts.head(N_TOP).to_string())

In [ ]:
top_objects = alert_counts.head(N_TOP).index.tolist()
df_top = df_gaia[df_gaia[COL_OBJ].isin(top_objects)].copy()
df_top[COL_MJD] = df_top[COL_MJD].astype(float)
print(f"Rows kept for top-{N_TOP} : {len(df_top):,}")

In [ ]:
df_top.head()

## 7. Utility functions

In [ ]:
def mjd_to_datestr(mjd_array):
    """Convert MJD array to ISO date strings 'YYYY-MM-DD'."""
    t = Time(np.asarray(mjd_array, dtype=float), format="mjd", scale="tai")
    return [tt.strftime("%Y-%m-%d") for tt in t]


def _add_date_axis(ax, dt_array, t0_mjd):
    """Secondary x-axis on top of *ax* showing calendar dates."""
    dt_finite = dt_array[np.isfinite(dt_array)]
    if len(dt_finite) == 0:
        return
    dt_min, dt_max = float(dt_finite.min()), float(dt_finite.max())
    if dt_max <= dt_min:
        tick_dt = np.array([dt_min])
    else:
        n_ticks = min(5, max(2, int((dt_max - dt_min) / 10) + 2))
        tick_dt = np.linspace(dt_min, dt_max, n_ticks)
    tick_dates = mjd_to_datestr(t0_mjd + tick_dt)
    ax2 = ax.twiny()
    ax2.set_xlim(ax.get_xlim())
    ax2.set_xticks(tick_dt)
    ax2.set_xticklabels(tick_dates, rotation=35, ha="left", fontsize=7)
    ax2.tick_params(axis="x", length=3)


def _shared_lim(arrays, margin=0.05):
    """Common [vmin, vmax] range from a list of arrays (with margin)."""
    all_vals = np.concatenate([np.asarray(a, dtype=float).ravel() for a in arrays])
    finite = all_vals[np.isfinite(all_vals)]
    if len(finite) == 0:
        return None
    vmin, vmax = finite.min(), finite.max()
    span = max(vmax - vmin, 1e-6)
    return vmin - margin * span, vmax + margin * span


def _object_color_palette(n_objects):
    """Return a list of n_objects distinguishable colours from tab20 / tab10."""
    cmap = cm.get_cmap("tab20" if n_objects > 10 else "tab10")
    return [cmap(i / max(n_objects - 1, 1)) for i in range(n_objects)]


print("Utility functions defined.")

## 7b. Per-object flux table — `get_diaobject_table()`

In [ ]:
def get_diaobject_table(diaObjectId, source_df=None) -> pd.DataFrame:
    """
    Return a flat analysis-ready DataFrame for a single diaObjectId.

    Parameters
    ----------
    diaObjectId : int or str
    source_df   : pd.DataFrame or None — defaults to df_gaia (Gaia-filtered)

    Returns
    -------
    pd.DataFrame with columns:
        diaObj, diaSrc, mjd, band, visit (Int64), detector (Int64),
        x, y, psfFlux, psfFluxErr,
        scienceFlux, scienceFluxErr,
        templateFlux, templateFluxErr,
        day_obs, month_obs
    """
    _src = source_df if source_df is not None else df_gaia
    mask = _src[COL_OBJ].astype(str) == str(diaObjectId)
    df_obj = _src[mask].copy()

    _empty_cols = [
        "diaObj",
        "diaSrc",
        "mjd",
        "band",
        "visit",
        "detector",
        "x",
        "y",
        "psfFlux",
        "psfFluxErr",
        "scienceFlux",
        "scienceFluxErr",
        "templateFlux",
        "templateFluxErr",
        "day_obs",
        "month_obs",
    ]
    if df_obj.empty:
        return pd.DataFrame(columns=_empty_cols)

    out = pd.DataFrame()
    out["diaObj"] = df_obj[COL_OBJ].values
    out["diaSrc"] = df_obj[COL_SRC].values
    out["mjd"] = pd.to_numeric(df_obj[COL_MJD], errors="coerce")
    out["band"] = df_obj[COL_BAND].values
    out["psfFlux"] = pd.to_numeric(df_obj[COL_PSF], errors="coerce")
    out["psfFluxErr"] = pd.to_numeric(df_obj[COL_PSFERR], errors="coerce")

    for col_in, col_out in (("r:visit", "visit"), ("r:detector", "detector")):
        if col_in in df_obj.columns:
            out[col_out] = pd.to_numeric(df_obj[col_in], errors="coerce").astype("Int64")
        else:
            out[col_out] = pd.array([pd.NA] * len(df_obj), dtype="Int64")

    for col_in, col_out in (("r:x", "x"), ("r:y", "y")):
        out[col_out] = pd.to_numeric(df_obj[col_in], errors="coerce") if col_in in df_obj.columns else np.nan

    out["scienceFlux"] = (
        pd.to_numeric(df_obj[COL_SCI], errors="coerce") if COL_SCI and COL_SCI in df_obj.columns else np.nan
    )
    out["scienceFluxErr"] = (
        pd.to_numeric(df_obj[COL_SCIERR], errors="coerce")
        if COL_SCIERR and COL_SCIERR in df_obj.columns
        else np.nan
    )
    out["templateFlux"] = (
        pd.to_numeric(df_obj[COL_TEMP], errors="coerce")
        if COL_TEMP and COL_TEMP in df_obj.columns
        else np.nan
    )
    out["templateFluxErr"] = (
        pd.to_numeric(df_obj[COL_TEMPERR], errors="coerce")
        if COL_TEMPERR and COL_TEMPERR in df_obj.columns
        else np.nan
    )

    out = (
        out[
            [
                "diaObj",
                "diaSrc",
                "mjd",
                "band",
                "visit",
                "detector",
                "x",
                "y",
                "psfFlux",
                "psfFluxErr",
                "scienceFlux",
                "scienceFluxErr",
                "templateFlux",
                "templateFluxErr",
            ]
        ]
        .sort_values(["mjd", "band"], na_position="last")
        .reset_index(drop=True)
    )
    out["day_obs"] = out["visit"] // 100000
    out["month_obs"] = out["visit"] // 10000000
    return out


print("get_diaobject_table() defined.")

### 7b-demo — Quick test on the highest-alert Gaia-matched object

In [ ]:
_test_oid = top_objects[0]
_test_meta = get_gaia_meta(_test_oid)
df_test = get_diaobject_table(_test_oid)

print(f"diaObjectId : {_test_oid}")
print(f"Gaia meta   : {_test_meta}")
print(f"  {len(df_test)} rows")
print(f"  bands            : {sorted(df_test['band'].dropna().unique())}")
print(f"  visit range      : {df_test['visit'].dropna().min()} … {df_test['visit'].dropna().max()}")
print(f"  mjd range        : {df_test['mjd'].min():.2f} … {df_test['mjd'].max():.2f}")
print(f"  scienceFlux NaN  : {df_test['scienceFlux'].isna().sum()} / {len(df_test)}")
print(f"  templateFlux NaN : {df_test['templateFlux'].isna().sum()} / {len(df_test)}")
display(df_test)

## 7b Load diaObject catalog  from file

In [ ]:
DIR_CAT = "data_FINK_BLOCK_LC_01"
_cat_priority = [
    os.path.join(DIR_CAT, "diaobj_catalogue_gaia_stable.csv"),
    os.path.join(DIR_CAT, "diaobj_catalogue.csv"),
]
df_cat_ref = None
for _cat_path in _cat_priority:
    if os.path.exists(_cat_path):
        df_cat_ref = pd.read_csv(_cat_path, low_memory=False)
        df_cat_ref["diaObjectId"] = df_cat_ref["diaObjectId"].astype(str)
        break

## 8. Per-object: psfFlux vs scienceFlux vs templateFlux

For each Gaia-matched DIA object (top-N by alert count): **7 subplots**
(one per band `u g r i z y` + combined).

Each panel shows:
- **●  filled circles** = `scienceFlux`  ± `scienceFluxErr`
- **□  open squares**   = `psfFlux`      ± `psfFluxErr`
- **+  plus markers**   = `templateFlux` ± `templateFluxErr`

The figure super-title includes the Gaia DR3 G magnitude, stability class,
and Fink group of the object.

In [ ]:
def plot_fluxes_one_object(obj_id, df_obj, src_label, rank):
    """
    Plot psfFlux, scienceFlux, and templateFlux for one Gaia-matched DIA object.

    Layout: 7 panels (u g r i z y | all-bands combined).
    Shared x and y scales across the 6 band panels.
    Figure title includes Gaia DR3 metadata.
    """

    # special with object catalog for flux straight line
    oid_str = str(obj_id)
    ref_psf = {}
    ref_sci = {}
    if df_cat_ref is not None:
        mask_cat = df_cat_ref["diaObjectId"].astype(str) == oid_str
        if mask_cat.any():
            cat_row = df_cat_ref[mask_cat].iloc[0]
            for b in BANDS:
                for dest, col in [(ref_psf, f"r:{b}_psfFluxMean"), (ref_sci, f"r:{b}_scienceFluxMean")]:
                    try:
                        fval = float(cat_row[col])
                        dest[b] = fval if np.isfinite(fval) else None
                    except (KeyError, TypeError, ValueError):
                        dest[b] = None

    meta = get_gaia_meta(obj_id)
    t0_mjd = df_obj[COL_MJD].min()
    t0_date = mjd_to_datestr([t0_mjd])[0]
    field = meta.get("field", df_obj["field"].iloc[0] if "field" in df_obj.columns else "")
    group = meta.get("group", "?")
    G_mag = meta.get("G_mag", np.nan)
    gaia_status = meta.get("gaia_status", "?")

    n_subplots = len(BANDS) + 1
    fig, axes = plt.subplots(
        1,
        n_subplots,
        figsize=(3.2 * n_subplots, 4.5),
        constrained_layout=True,
    )

    band_data = {}
    all_flux = []
    all_dt = []

    for band in BANDS:
        mask = df_obj[COL_BAND] == band
        df_b = df_obj[mask].sort_values(COL_MJD)
        if len(df_b) == 0:
            band_data[band] = None
            continue
        dt = df_b[COL_MJD].values - t0_mjd
        psf = df_b[COL_PSF].values.astype(float)
        psf_err = df_b[COL_PSFERR].values.astype(float)
        sci = df_b[COL_SCI].values.astype(float) if has_science else None
        sci_err = df_b[COL_SCIERR].values.astype(float) if has_sci_err else None
        temp = df_b[COL_TEMP].values.astype(float) if has_template else None
        temp_err = df_b[COL_TEMPERR].values.astype(float) if has_template_err else None

        band_data[band] = (dt, psf, psf_err, sci, sci_err, temp, temp_err)
        all_dt.append(dt)
        all_flux.append(psf)
        if sci is not None:
            all_flux.append(sci)
        if temp is not None:
            all_flux.append(temp)

    ylim = _shared_lim(all_flux) if all_flux else None
    xlim = _shared_lim(all_dt) if all_dt else None

    # loop on bands subplots
    for idx, band in enumerate(BANDS):
        ax = axes[idx]
        color = BAND_COLORS.get(band, "k")

        if band_data[band] is None:
            ax.text(
                0.5,
                0.5,
                "no data",
                ha="center",
                va="center",
                transform=ax.transAxes,
                color="gray",
                fontsize=8,
            )
            ax.set_title(f"band {band}", fontsize=9)
            if xlim:
                ax.set_xlim(xlim)
            if ylim:
                ax.set_ylim(ylim)
            ax.set_xlabel("Δt [days]", fontsize=8)
            continue

        dt, psf, psf_err, sci, sci_err, temp, temp_err = band_data[band]

        if sci is not None:
            ax.errorbar(
                dt,
                sci,
                yerr=sci_err,
                fmt="o",
                ms=5,
                color=color,
                ecolor=color,
                elinewidth=0.9,
                capsize=2,
                alpha=0.90,
                label="scienceFlux",
            )

        ax.errorbar(
            dt,
            psf,
            yerr=psf_err,
            fmt="s",
            ms=4,
            color=color,
            ecolor=color,
            elinewidth=0.7,
            capsize=2,
            alpha=0.55,
            mfc="none",
            label="psfFlux",
        )

        if temp is not None:
            ax.errorbar(
                dt,
                temp,
                yerr=temp_err,
                fmt="+",
                ms=5,
                color=color,
                ecolor=color,
                elinewidth=0.7,
                capsize=2,
                alpha=0.55,
                mfc="none",
                label="templateFlux",
            )

        # add object flux per band
        v_sci = ref_sci.get(band)
        if v_sci is not None:
            ax.axhline(v_sci, color=color, lw=1.2, ls="-.", alpha=0.85, label="cat sciFluxMean")
        v_psf = ref_psf.get(band)
        if v_psf is not None:
            ax.axhline(v_psf, color=color, lw=1.2, ls=":", alpha=0.85, label="cat psfFluxMean")

        ax.axhline(0.0, color="k", lw=0.6, ls=":", alpha=0.4)
        ax.set_title(f"band {band}", fontsize=9)
        ax.set_ylabel("flux [nJy]", fontsize=8)
        ax.set_xlabel("Δt [days]", fontsize=8)
        ax.legend(fontsize=7, loc="best")
        if xlim:
            ax.set_xlim(xlim)
        if ylim:
            ax.set_ylim(ylim)
        _add_date_axis(ax, dt, t0_mjd)

    # Combined panel
    ax_all = axes[-1]
    for band in BANDS:
        if band_data[band] is None:
            continue
        dt, psf, psf_err, sci, sci_err, temp, temp_err = band_data[band]
        color = BAND_COLORS.get(band, "k")
        if sci is not None:
            ax_all.errorbar(
                dt,
                sci,
                yerr=sci_err,
                fmt="o",
                ms=3,
                color=color,
                ecolor=color,
                elinewidth=0.7,
                capsize=2,
                alpha=0.80,
                label=f"{band} sci",
            )
        ax_all.errorbar(
            dt,
            psf,
            yerr=psf_err,
            fmt="s",
            ms=3,
            color=color,
            ecolor=color,
            elinewidth=0.6,
            capsize=2,
            alpha=0.45,
            mfc="none",
            label=f"{band} psf",
        )
        if temp is not None:
            ax_all.errorbar(
                dt,
                temp,
                yerr=temp_err,
                fmt="+",
                ms=3,
                color=color,
                ecolor=color,
                elinewidth=0.7,
                capsize=2,
                alpha=0.80,
                mfc="none",
                label=f"{band} temp",
            )

        # horizontal lines with object flux
        v_sci = ref_sci.get(band)
        if v_sci is not None:
            ax_all.axhline(v_sci, color=color, lw=0.7, ls="-.", alpha=0.45, label="_nolegend_")
        v_psf = ref_psf.get(band)
        if v_psf is not None:
            ax_all.axhline(v_psf, color=color, lw=0.7, ls=":", alpha=0.45, label="_nolegend_")

    ax_all.axhline(0.0, color="k", lw=0.6, ls=":", alpha=0.4)
    ax_all.set_title("all bands — sci(●) psf(□) temp(+)", fontsize=9)
    ax_all.set_ylabel("flux [nJy]", fontsize=8)
    ax_all.set_xlabel("Δt [days]", fontsize=8)
    ax_all.legend(fontsize=6, ncol=4, loc="best")
    if xlim:
        ax_all.set_xlim(xlim)
    _add_date_axis(ax_all, df_obj[COL_MJD].values - t0_mjd, t0_mjd)

    G_str = f"{G_mag:.2f}" if np.isfinite(G_mag) else "?"
    fig.suptitle(
        f"rank #{rank + 1}  |  diaObjectId={obj_id}  |  field={field}  |  N={len(df_obj)}  |  t₀={t0_date}\n"
        f"Fink group: {group}   |   Gaia G={G_str} mag   |   Gaia status: {gaia_status}\n"
        "scienceFlux (●) vs psfFlux (□) vs templateFlux (+)   [nJy, no normalisation]",
        fontsize=10,
        y=1.09,
    )
    savefig(f"fluxes_gaia_obj_{src_label}_rank{rank + 1:02d}_obj{obj_id}")
    plt.show()


print("plot_fluxes_one_object() defined.")

In [ ]:
if not has_science:
    print("scienceFlux column not available — Section 8 skipped.")
else:
    for rank, obj_id in enumerate(top_objects):
        df_obj = df_top[df_top[COL_OBJ] == obj_id].sort_values(COL_MJD)
        plot_fluxes_one_object(obj_id, df_obj, src_label, rank)

## 9. Per-band multi-object overview

One figure per band (+ combined): all top-N Gaia-matched objects plotted on
the same axes with a distinct colour per object.

In [ ]:
obj_colors = _object_color_palette(len(top_objects))

for band_sel in BANDS + ["all"]:
    fig, ax = plt.subplots(figsize=(11, 4), constrained_layout=True)

    for o_idx, obj_id in enumerate(top_objects):
        meta = get_gaia_meta(obj_id)
        group = meta.get("group", "?")
        G_str = f"G={meta.get('G_mag', np.nan):.1f}" if np.isfinite(meta.get("G_mag", np.nan)) else "G=?"
        obj_color = obj_colors[o_idx]

        df_obj = df_top[df_top[COL_OBJ] == obj_id].sort_values(COL_MJD)
        t0_mjd = df_obj[COL_MJD].min()

        if band_sel == "all":
            df_b = df_obj
        else:
            df_b = df_obj[df_obj[COL_BAND] == band_sel]
        if len(df_b) == 0:
            continue

        dt = df_b[COL_MJD].values.astype(float) - t0_mjd
        psf = df_b[COL_PSF].values.astype(float)
        psf_err = df_b[COL_PSFERR].values.astype(float)
        sci = df_b[COL_SCI].values.astype(float) if has_science else None
        sci_err = df_b[COL_SCIERR].values.astype(float) if has_sci_err else None

        label_base = f"obj{o_idx + 1} {G_str} ({group})"

        if sci is not None:
            ax.errorbar(
                dt,
                sci,
                yerr=sci_err,
                fmt="o",
                ms=3,
                color=obj_color,
                ecolor=obj_color,
                elinewidth=0.6,
                capsize=1.5,
                alpha=0.85,
                label=label_base + " sci",
            )
        ax.errorbar(
            dt,
            psf,
            yerr=psf_err,
            fmt="s",
            ms=2.5,
            color=obj_color,
            ecolor=obj_color,
            elinewidth=0.5,
            capsize=1,
            alpha=0.45,
            mfc="none",
            label=label_base + " psf",
        )

    ax.axhline(0.0, color="k", lw=0.6, ls=":", alpha=0.4)
    band_lbl = band_sel if band_sel != "all" else "all bands"
    ax.set_title(
        f"Gaia-matched objects — {band_lbl}  |  scienceFlux (●) vs psfFlux (□)  [nJy]",
        fontsize=10,
    )
    ax.set_xlabel("Δt [days from object t₀]", fontsize=9)
    ax.set_ylabel("flux [nJy]", fontsize=9)
    ax.legend(fontsize=6, ncol=4, loc="best")

    fname = f"multiobj_band{band_sel}_{src_label}"
    savefig(fname)
    plt.show()

## 10. Summary statistics: median flux levels

For each Gaia-matched object and band, report the median `scienceFlux`,
`psfFlux`, and `templateFlux`, plus the ratio
`median(psfFlux) / median(scienceFlux)`.  This should be ~0 for a stable
star whose template was built from images of the same star.

In [ ]:
records = []

for rank, obj_id in enumerate(top_objects):
    df_obj = df_top[df_top[COL_OBJ] == obj_id]
    meta = get_gaia_meta(obj_id)
    t0_mjd = df_obj[COL_MJD].min()
    t0_date = mjd_to_datestr([t0_mjd])[0]

    row = {
        "rank": rank + 1,
        "diaObjectId": obj_id,
        "n_alerts": len(df_obj),
        "t0_date": t0_date,
        "fink_group": meta.get("group", "?"),
        "gaia_G_mag": meta.get("G_mag", np.nan),
        "gaia_status": meta.get("gaia_status", "?"),
        "dr3_source_id": meta.get("dr3_id", None),
    }

    for band in BANDS:
        df_b = df_obj[df_obj[COL_BAND] == band]
        if len(df_b) == 0:
            row[f"med_sci_{band}"] = np.nan
            row[f"med_psf_{band}"] = np.nan
            row[f"med_temp_{band}"] = np.nan
            row[f"ratio_psf_sci_{band}"] = np.nan
            continue

        med_psf = float(np.nanmedian(df_b[COL_PSF].values.astype(float)))
        row[f"med_psf_{band}"] = round(med_psf, 2)

        if has_science:
            med_sci = float(np.nanmedian(df_b[COL_SCI].values.astype(float)))
            row[f"med_sci_{band}"] = round(med_sci, 2)
            row[f"ratio_psf_sci_{band}"] = round(med_psf / med_sci, 4) if med_sci != 0 else np.nan
        else:
            row[f"med_sci_{band}"] = np.nan
            row[f"ratio_psf_sci_{band}"] = np.nan

        if has_template:
            row[f"med_temp_{band}"] = round(float(np.nanmedian(df_b[COL_TEMP].values.astype(float))), 2)
        else:
            row[f"med_temp_{band}"] = np.nan

    records.append(row)

df_stats = pd.DataFrame(records)
print("Median flux levels per Gaia-matched object and band [nJy]:")
print("  med_sci   = median(scienceFlux)")
print("  med_psf   = median(psfFlux)")
print("  med_temp  = median(templateFlux)")
print("  ratio     = med_psf / med_sci  (∼0 expected for stable star in good template)")
print()
display(df_stats)

In [ ]:
out_path = os.path.join(DIR_FIGS, f"median_flux_stats_gaia_{src_label}.csv")
df_stats.to_csv(out_path, index=False)
print(f"Saved → {out_path}")

## 11. Summary

This notebook is the Gaia DR3–restricted version of
`07_psfFlux_scienceFlux_templateFlux.ipynb`.  It selects the
subset of Fink diaObjects that were matched to a Gaia DR3 source
by notebook `08b`, and displays the three raw flux quantities
(`psfFlux`, `scienceFlux`, `templateFlux`) with the Gaia stability
and G-magnitude context in each figure title.

| Section | Content |
|---------|----------|
| 2       | Load src parquet (same as notebook 07) |
| 3       | Load Gaia matched catalogue from 08b |
| 5       | Filter parquet to Gaia-matched diaObjectIds |
| 6       | Rank by alert count |
| 8       | Per-object flux plots (7 panels each) |
| 9       | Per-band multi-object overview |
| 10      | Summary statistics table (CSV) |

**Outputs:** figures in `figs_FINK_BLOCK_LC_09/` and CSV stats table.
